In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
TICKETS_PATH = "../data/raw/support_tickets.csv"

tickets = pd.read_csv(TICKETS_PATH)

print("Dataset shape:", tickets.shape)
print(tickets.head())

Dataset shape: (10000, 15)
   ticket_id created_date                                        ticket_text  \
0  TKT100000   2023-08-12  The lipstick I received in order #ORD5614226 s...   
1  TKT100001   2023-12-11  The shoe rack received in order #ORD8038374 ha...   
2  TKT100002   2024-01-06  I want to exchange my conditioner (order #ORD6...   
3  TKT100003   2024-11-17  My makeup brush set from order #ORD4823498 app...   
4  TKT100004   2023-08-23  Product listing for lipstick is completely mis...   

       product_name         product_segment  ticket_category priority  \
0          lipstick  Beauty & Personal Care    Product Issue     High   
1         shoe rack          Home & Kitchen    Product Issue     High   
2       conditioner  Beauty & Personal Care  Refund & Return      Low   
3  makeup brush set  Beauty & Personal Care    Product Issue     High   
4          lipstick  Beauty & Personal Care    Product Issue   Medium   

       sentiment  confidence_score escalated  \
0    

In [3]:
sentiment_df = tickets[[
    "ticket_text",
    "sentiment"
]].copy()

sentiment_df["clean_text"] = (
    sentiment_df["ticket_text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Original records:", len(sentiment_df))

sentiment_df = sentiment_df.drop_duplicates(
    subset="clean_text"
).reset_index(drop=True)

print("After deduplication:", len(sentiment_df))
print("Duplicates removed:", len(tickets) - len(sentiment_df))

Original records: 10000
After deduplication: 8777
Duplicates removed: 1223


In [4]:
print("\nSentiment distribution:")
print(sentiment_df["sentiment"].value_counts())

print("\nSentiment percentages:")
print(
    sentiment_df["sentiment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Sentiment distribution:
sentiment
Negative             2875
Neutral              2274
Slightly Negative    1508
Very Negative        1445
Positive              675
Name: count, dtype: int64

Sentiment percentages:
sentiment
Negative             32.76
Neutral              25.91
Slightly Negative    17.18
Very Negative        16.46
Positive              7.69
Name: proportion, dtype: float64


In [5]:
X = sentiment_df["clean_text"]
y = sentiment_df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 7021
Testing samples : 1756


In [6]:
sentiment_tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = sentiment_tfidf.fit_transform(X_train)
X_test_tfidf = sentiment_tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape :", X_test_tfidf.shape)

Training TF-IDF shape: (7021, 3128)
Testing TF-IDF shape : (1756, 3128)


In [7]:
sentiment_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

sentiment_model.fit(
    X_train_tfidf,
    y_train
)

print("Sentiment model training completed.")

Sentiment model training completed.


In [8]:
y_pred = sentiment_model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test,
    y_pred,
    average="weighted"
)
recall = recall_score(
    y_test,
    y_pred,
    average="weighted"
)
f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print("=" * 60)
print("SENTIMENT MODEL PERFORMANCE")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

SENTIMENT MODEL PERFORMANCE
Accuracy : 0.4618
Precision: 0.4871
Recall   : 0.4618
F1 Score : 0.4451

Classification Report:
                   precision    recall  f1-score   support

         Negative       0.60      0.50      0.55       575
          Neutral       0.49      0.18      0.27       455
         Positive       0.43      0.73      0.54       135
Slightly Negative       0.46      0.71      0.56       302
    Very Negative       0.31      0.44      0.36       289

         accuracy                           0.46      1756
        macro avg       0.46      0.51      0.45      1756
     weighted avg       0.49      0.46      0.45      1756



In [16]:
from sklearn.svm import LinearSVC

sentiment_svm = LinearSVC(
    class_weight="balanced",
    random_state=42
)

sentiment_svm.fit(
    X_train_tfidf,
    y_train
)

y_pred_svm = sentiment_svm.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred_svm)
precision = precision_score(
    y_test,
    y_pred_svm,
    average="weighted"
)
recall = recall_score(
    y_test,
    y_pred_svm,
    average="weighted"
)
f1 = f1_score(
    y_test,
    y_pred_svm,
    average="weighted"
)

print("=" * 60)
print("SENTIMENT - LINEAR SVM PERFORMANCE")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_svm
    )
)

SENTIMENT - LINEAR SVM PERFORMANCE
Accuracy : 0.4909
Precision: 0.4787
Recall   : 0.4909
F1 Score : 0.4786

Classification Report:
                   precision    recall  f1-score   support

         Negative       0.60      0.69      0.64       575
          Neutral       0.48      0.35      0.40       455
         Positive       0.42      0.59      0.49       135
Slightly Negative       0.47      0.55      0.50       302
    Very Negative       0.29      0.21      0.25       289

         accuracy                           0.49      1756
        macro avg       0.45      0.48      0.46      1756
     weighted avg       0.48      0.49      0.48      1756



In [9]:
# Test with real tickets
test_tickets = [
    "I am very unhappy with this service",
    "My order arrived on time",
    "The delivery is slightly delayed",
    "This is the worst experience I have ever had",
    "I am not sure when my order will arrive"
]

test_vectors = sentiment_tfidf.transform(test_tickets)

predictions = sentiment_model.predict(test_vectors)
probabilities = sentiment_model.predict_proba(test_vectors)

print("=" * 60)
print("SENTIMENT PREDICTIONS")
print("=" * 60)

for text, prediction, probability in zip(
    test_tickets,
    predictions,
    probabilities
):
    confidence = probability.max()

    print(f"\nTicket: {text}")
    print(f"Sentiment: {prediction}")
    print(f"Confidence: {confidence:.4f}")

SENTIMENT PREDICTIONS

Ticket: I am very unhappy with this service
Sentiment: Slightly Negative
Confidence: 0.2729

Ticket: My order arrived on time
Sentiment: Negative
Confidence: 0.2610

Ticket: The delivery is slightly delayed
Sentiment: Neutral
Confidence: 0.4446

Ticket: This is the worst experience I have ever had
Sentiment: Slightly Negative
Confidence: 0.2683

Ticket: I am not sure when my order will arrive
Sentiment: Positive
Confidence: 0.3033


In [10]:
sentiment_priority = pd.crosstab(
    sentiment_df["sentiment"],
    tickets.loc[sentiment_df.index, "priority"]
)

print(sentiment_priority)

priority           High  Low  Medium
sentiment                           
Negative           1341  488    1046
Neutral            1035  395     844
Positive            303  125     247
Slightly Negative   688  250     570
Very Negative       661  254     530


In [11]:
sentiment_priority_df = tickets[[
    "ticket_text",
    "sentiment",
    "priority"
]].copy()

sentiment_priority_df["clean_text"] = (
    sentiment_priority_df["ticket_text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

sentiment_priority_df = sentiment_priority_df.drop_duplicates(
    subset="clean_text"
)

print(
    pd.crosstab(
        sentiment_priority_df["sentiment"],
        sentiment_priority_df["priority"]
    )
)

priority           High  Low  Medium
sentiment                           
Negative           2875    0       0
Neutral               0  701    1573
Positive              0  675       0
Slightly Negative     0    0    1508
Very Negative      1445    0       0


In [13]:
sentiment_analysis_df = tickets[[
    "ticket_text",
    "sentiment",
    "priority",
    "ticket_category"
]].copy()

sentiment_analysis_df["clean_text"] = (
    sentiment_analysis_df["ticket_text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

sentiment_analysis_df = sentiment_analysis_df.drop_duplicates(
    subset="clean_text"
)

print("Shape:", sentiment_analysis_df.shape)

print("\nSentiment vs Priority:")
print(
    pd.crosstab(
        sentiment_analysis_df["sentiment"],
        sentiment_analysis_df["priority"]
    )
)

print("\nSentiment vs Category:")
print(
    pd.crosstab(
        sentiment_analysis_df["sentiment"],
        sentiment_analysis_df["ticket_category"]
    )
)

Shape: (8777, 5)

Sentiment vs Priority:
priority           High  Low  Medium
sentiment                           
Negative           2875    0       0
Neutral               0  701    1573
Positive              0  675       0
Slightly Negative     0    0    1508
Very Negative      1445    0       0

Sentiment vs Category:
ticket_category    Account & Login  App & Website Issue  Delivery Issue  \
sentiment                                                                 
Negative                        10                   13             664   
Neutral                         23                  278             492   
Positive                         9                  176              87   
Slightly Negative               19                   89             362   
Very Negative                    7                   13             345   

ticket_category    Payment Issue  Product Issue  Refund & Return  \
sentiment                                                          
Negative      

In [15]:
print(
    pd.crosstab(
        sentiment_analysis_df["sentiment"],
        sentiment_analysis_df["ticket_category"]
    )
)

ticket_category    Account & Login  App & Website Issue  Delivery Issue  \
sentiment                                                                 
Negative                        10                   13             664   
Neutral                         23                  278             492   
Positive                         9                  176              87   
Slightly Negative               19                   89             362   
Very Negative                    7                   13             345   

ticket_category    Payment Issue  Product Issue  Refund & Return  \
sentiment                                                          
Negative                     502            718              659   
Neutral                      333            439              533   
Positive                     113             81              192   
Slightly Negative            255            296              330   
Very Negative                211            392              337  

In [17]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

e:\HCL Guvi\Multi-Agent Customer Support\Multi-Agent Customer Support Platform\.supportvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cpu
Transformers: 5.18.0


In [18]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Pretrained sentiment model loaded successfully.")

e:\HCL Guvi\Multi-Agent Customer Support\Multi-Agent Customer Support Platform\.supportvenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rucha\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\HCL Guvi\Multi-Agent Custom

Pretrained sentiment model loaded successfully.


In [19]:
test_tickets = [
    "I am very unhappy with this service",
    "My order arrived on time",
    "The delivery is slightly delayed",
    "This is the worst experience I have ever had",
    "I am not sure when my order will arrive"
]

results = sentiment_pipeline(test_tickets)

for text, result in zip(test_tickets, results):
    print("\nTicket:", text)
    print("Prediction:", result["label"])
    print("Confidence:", round(result["score"], 4))


Ticket: I am very unhappy with this service
Prediction: NEGATIVE
Confidence: 0.9997

Ticket: My order arrived on time
Prediction: POSITIVE
Confidence: 0.6885

Ticket: The delivery is slightly delayed
Prediction: NEGATIVE
Confidence: 0.9989

Ticket: This is the worst experience I have ever had
Prediction: NEGATIVE
Confidence: 0.9998

Ticket: I am not sure when my order will arrive
Prediction: NEGATIVE
Confidence: 0.9995


In [20]:
sentiment_df["binary_sentiment"] = sentiment_df["sentiment"].apply(
    lambda x: "POSITIVE"
    if x == "Positive"
    else "NEGATIVE"
)

In [21]:
from sklearn.metrics import accuracy_score, classification_report

# Use the same test split
test_texts = X_test.tolist()

# Pretrained model predictions
transformer_results = sentiment_pipeline(
    test_texts,
    batch_size=16,
    truncation=True
)

predicted_binary_sentiment = [
    result["label"]
    for result in transformer_results
]

# Convert dataset labels to binary
actual_binary_sentiment = y_test.apply(
    lambda x: (
        "POSITIVE"
        if x == "Positive"
        else "NEGATIVE"
        if x in ["Negative", "Slightly Negative", "Very Negative"]
        else "NEUTRAL"
    )
)

# Exclude Neutral for binary evaluation
mask = actual_binary_sentiment != "NEUTRAL"

y_actual_binary = actual_binary_sentiment[mask]
y_pred_binary = np.array(predicted_binary_sentiment)[mask]

print("=" * 60)
print("PRETRAINED SENTIMENT MODEL")
print("BINARY EVALUATION")
print("=" * 60)

print("Evaluation samples:", len(y_actual_binary))

print(
    "Accuracy:",
    accuracy_score(
        y_actual_binary,
        y_pred_binary
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_actual_binary,
        y_pred_binary
    )
)

PRETRAINED SENTIMENT MODEL
BINARY EVALUATION
Evaluation samples: 1301
Accuracy: 0.9046887009992314

Classification Report:
              precision    recall  f1-score   support

    NEGATIVE       0.90      1.00      0.95      1166
    POSITIVE       0.92      0.09      0.16       135

    accuracy                           0.90      1301
   macro avg       0.91      0.54      0.56      1301
weighted avg       0.91      0.90      0.87      1301

